# 06 · Submission

The test series are predicted on Kaggle. It is a code competition: a notebook runs `pe.inference` offline on two T4 GPUs and four CPUs within 9 hours, from a dataset that holds the code and the weights. The scores are late submissions; the private leaderboard scores about 72% of the hidden test series, the public one the rest.

**Contents**
1. Slice models alone
2. Series models
3. Leaderboard

Needs `slices.csv` and `series.csv` (02), the slice runs of 04 (`outputs/runs/`, `models/`), the series runs and features of 05 (`outputs/series_runs/`, `outputs/features/`) and the DICOM files of a few series for section 2.

In [1]:
import json
import re
import subprocess
import sys

import numpy as np
import pandas as pd
import torch

from pe.data.prepare import read_slices
from pe.inference import SeriesPredictor, prepare_study
from pe.models.series_model import SeriesModel
from pe.paths import REPO_ROOT, load_paths
from pe.series_training import SliceFeatures, with_neighbours

paths = load_paths()
OUTPUTS = paths["outputs_dir"]
FINAL = ("s7_noneigh_f{fold}_s1", "s6_selogit_f{fold}_s1", "s7_cnxemb_nn_f{fold}_s1")  # SE embedding [f], SE logit, ConvNeXt embedding [f] (05)
FINAL_LOCAL = 0.1955  # test-like, out of fold (05)
# private and public scores, read from Kaggle's submissions page
LEADERBOARD = {"se_f0": (0.244, 0.249), "convnext_f0": (0.236, 0.245), "both_f0": (0.233, 0.241),
               "se_5folds": (0.228, 0.234), "convnext_5folds": (0.228, 0.237), "both_5folds": (0.225, 0.232),
               "final_f0": (0.152, 0.152), "final_5folds": (0.144, 0.143)}

## 1. Slice models alone

A submission without the series models needs two things the slice models do not give: calibrated slice probabilities and the nine series labels.

- **Slices:** one weighted logistic regression per variant, a + b · (mean logit of each backbone). It is fitted on the out-of-fold logits of the test-like series with the metric's slice weight, so only the slices of series with PE count. It corrects the level and sets the weight of each backbone.
- **Series labels:** the training label rates, the same for every series.
- Slices outside the main stack get 0.001, then the consistency fix (03) runs; it raises only main-stack slices.

Six variants: SE-ResNeXt101, ConvNeXt Tiny or both, with the fold-0 models or all five folds (the test gets the mean of a backbone's five logits).

The scores are test-like and out of sample. **raw** takes the mean logit as it is, **level only** fits the constant a, **regression** fits a and b (the submitted one). The five-fold variants fit on four folds and score the fifth with that fold's models; the fold-0 variants fit on half of fold 0's series and score the other half. The last column is the fit on all the data, which the test set gets.

In [2]:
printed = subprocess.run([sys.executable, "scripts/fit_slice_ensemble.py"], cwd=REPO_ROOT, capture_output=True, text=True, check=True).stdout
print(printed)
LOCAL = {name: float(regression) for name, regression in re.findall(r"^(\w+)\s+[\d.]+\s+[\d.]+\s+([\d.]+)", printed, re.M)}
assert set(LOCAL) == set(LEADERBOARD) - {"final_f0", "final_5folds"}, LOCAL

                    raw  level only  regression                                         fitted on all
variant                                                                                              
se_f0            0.2913      0.2890      0.2719                          0.736 + 1.771 x seresnext101
convnext_f0      0.3034      0.2817      0.2665                         0.760 + 0.608 x convnext_tiny
both_f0          0.2771      0.2639      0.2636  0.850 + 0.773 x seresnext101 + 0.380 x convnext_tiny
se_5folds        0.2876      0.2836      0.2680                          1.021 + 1.779 x seresnext101
convnext_5folds  0.3306      0.2827      0.2694                         1.213 + 0.626 x convnext_tiny
both_5folds      0.2913      0.2648      0.2632  1.227 + 1.001 x seresnext101 + 0.310 x convnext_tiny



### 1.1 Key findings

- **The regression gains 0.014 to 0.061 over the raw logits;** most for ConvNeXt over five folds (0.3306 to 0.2694).
- **The two backbones need opposite corrections:** SE's logits are too flat (slope 1.77), ConvNeXt's too steep (0.61).
- **Both backbones together score best:** 0.2636 with fold 0, 0.2632 with five folds.

## 2. Series models

The final of 05: three series models (SE-ResNeXt101 embedding [f], SE-ResNeXt101 logit [f, f − previous, f − next], ConvNeXt Tiny embedding [f]), each trained per fold on the features of that fold's slice models. On the test set, per series:

1. The two slice models of fold k read the main-stack slices in float16 and give the features: SE's 2,048 embedding values and its logit, ConvNeXt's 768 embedding values.
2. The three series models of fold k read them. Their probabilities, of the series and of every slice, are averaged over the three models and, for `final_5folds`, over the five folds.
3. Slices outside the main stack get 0.001, then one consistency fix.

`final_f0` runs fold 0 (2 slice and 3 series models), `final_5folds` all five (10 and 15). Each GPU holds every model and takes whole series from one queue, so both T4s work the whole run.

The check runs the submission code for fold 0 from the DICOM files of the series on this machine and compares its probabilities, before the fix, with the same series models on 05's saved features (computed in bfloat16).

In [3]:
slices = read_slices(paths["processed_data_dir"] / "slices.csv")
ensemble = json.loads((OUTPUTS / "submission" / "ensemble.json").read_text())
slice_files = {backbone: {k: paths["models_dir"] / f"{run}_e1.pt" for k, run in enumerate(runs)} for backbone, runs in ensemble["runs"].items()}
series_runs = [[OUTPUTS / "series_runs" / name.format(fold=k) for k in range(5)] for name in FINAL]
predictor = SeriesPredictor(slice_files, series_runs, [0], "cuda")

saved = []  # the same series models on the saved features of fold 0
for runs in series_runs:
    config = json.loads((runs[0] / "config.json").read_text())
    features = SliceFeatures(OUTPUTS / "features", 0, config["backbones"].split(","), config["input"])
    neighbours = not config["no_neighbours"]
    model = SeriesModel(sum(features.widths) * (3 if neighbours else 1), config["hidden"], 9, config["dropout"])
    model.load_state_dict(torch.load(runs[0] / "model.pt", map_location="cpu"))
    saved.append((model.eval().cuda(), features, neighbours))

local = sorted(d for split in ("train", "test") for d in (paths["raw_data_dir"] / split).iterdir())
d_series, d_image = [], []
for study_dir in local:
    item = prepare_study(study_dir)
    table = slices[(slices["series"] == study_dir.name) & slices["main"]].reset_index(drop=True)
    n = len(item["crops"])
    assert list(item["image"][:n]) == list(table["image"]), f"{study_dir.name}: main-stack order differs"
    p_series, p_image = predictor(item["crops"])
    want_series, want_image = np.zeros(9), np.zeros(n)
    with torch.inference_mode():
        for model, features, neighbours in saved:
            x = torch.from_numpy(features[features.rows_of(table)]).float().cuda()[None]
            s, e = model(with_neighbours(x) if neighbours else x, torch.tensor([n]))
            want_image += torch.sigmoid(s[0]).cpu().numpy() / len(saved)
            want_series += torch.sigmoid(e[0]).cpu().numpy() / len(saved)
    d_series.append(np.abs(p_series - want_series))
    d_image.append(np.abs(p_image - want_image))
d_series, d_image = np.concatenate(d_series), np.concatenate(d_image)
print(f"{len(local)} series, {len(d_image):,} main-stack slices; difference to the saved features, mean and max:")
print(f"  series labels {d_series.mean():.4f} {d_series.max():.4f} | slices {d_image.mean():.4f} {d_image.max():.4f}")

30 series, 7,595 main-stack slices; difference to the saved features, mean and max:
  series labels 0.0006 0.0044 | slices 0.0004 0.0061


### 2.1 Key findings

- **The submission code reproduces 05's series models from the DICOM files:** on 30 series the probabilities differ by 0.0044 at most for the series labels and 0.0061 for the slices. The slice models run in float16 there and ran in bfloat16 for the saved features.

## 3. Leaderboard

All eight submissions. **local** is the test-like score of section 1 (regression) and of 05's final, out of fold: each series is scored by one fold's models. It cannot see the mean of the five folds, so both final variants get 05's 0.1955.

In [4]:
board = pd.DataFrame(LEADERBOARD, index=["private", "public"]).T
board.insert(0, "local", pd.Series({**LOCAL, "final_f0": FINAL_LOCAL, "final_5folds": FINAL_LOCAL}))
board.sort_values("private")

,local,private,public
final_5folds,0.1955,0.144,0.143
final_f0,0.1955,0.152,0.152
both_5folds,0.2632,0.225,0.232
se_5folds,0.2680,0.228,0.234
convnext_5folds,0.2694,0.228,0.237
both_f0,0.2636,0.233,0.241
convnext_f0,0.2665,0.236,0.245
se_f0,0.2719,0.244,0.249


### 3.1 Key findings

- **The series models are the largest step: 0.225 to 0.144 private** with five folds; locally 0.2632 to 0.1955.
- **The mean of the five folds gains 0.008 for the final** (0.152 to 0.144) and 0.008 to 0.016 for the slice models alone; the local scores cannot measure it.
- **The local scores order the backbones as the leaderboard does:** both best, then ConvNeXt, then SE with fold 0; with five folds SE and ConvNeXt tie on the leaderboard (0.228).
- **Private and public differ by 0.009 at most.** Every choice was made on the local score before any submission.
- **The leaderboard is about 0.04 lower than the local score** (0.1955 to 0.152 for fold 0's models), likely because the test series were labelled by three readers and the training series by one (00).
- **final_5folds would place 2nd of 784 teams on the private leaderboard** (1st 0.140, 2nd 0.148); final_f0 matches the 4th place score, the best slice-only submission would be 66th. As a late submission it had six years more: ConvNeXt (2022) and the published solutions, whose series model settings it uses.

## Summary

**Result:** the three series models of 05 over the five folds score 0.144 private and 0.143 public, against 0.225 and 0.232 for the slice models alone.

**Limits**
- The mean of the five folds is measured only by the leaderboard.
- The uncertainty from the finite test set is not estimated; private and public, two parts of it, differ by 0.009 at most.
- Kaggle shows no running times for the hidden test; all eight finished within the 9 hours.